# ______ЭТАП I______

## ЯЧЕЙКА 1 — Установка зависимостей ##

In [ ]:
!pip install -q spacy stanza openpyxl
!python -m spacy download en_core_web_trf

import stanza
stanza.download(lang="ru", processors="tokenize,pos,lemma,depparse")
print("✓ Все модели загружены")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 31.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 25.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading these customized packages for language: ru (Russian)...
| Processor       | Package            |
----------------------------------------
| tokenize        | syntagrus          |
| pos             | syntagrus_charlm   |
| lemma           | syntagrus_nocharlm |
| depparse        | syntagrus_charlm   |
| backward_charlm | newswiki           |
| forward_charlm  | newswiki           |
| pretrain        | conll17            |



INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/tokenize/syntagrus.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/pos/syntagrus_charlm.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/lemma/syntagrus_nocharlm.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/depparse/syntagrus_charlm.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/backward_charlm/newswiki.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/forward_charlm/newswiki.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/pretrain/conll17.pt
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


✓ Все модели загружены


## ЯЧЕЙКА 2 — Импорты и инициализация моделей ##

In [ ]:
import spacy, stanza, openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from google.colab import files

print("Загрузка spaCy en_core_web_trf ...")
nlp_en = spacy.load("en_core_web_trf")

print("Загрузка Stanza RU (ru_syntagrus) ...")
nlp_ru = stanza.Pipeline(
    lang="ru",
    processors="tokenize,pos,lemma,depparse",
    tokenize_no_ssplit=True,  # весь concept_unit = одно предложение
    verbose=False,
)
print("✓ Обе модели готовы")

Загрузка spaCy en_core_web_trf ...
Загрузка Stanza RU (ru_syntagrus) ...
✓ Обе модели готовы


## ЯЧЕЙКА 3 — Функции построения строки UD-структуры ##

In [ ]:
def _fmt_deprel(label: str) -> str:
    """Нормализует deprel: сохраняет подтип только для значимых меток."""
    keep = {"nsubj", "obj", "obl", "nmod", "acl", "advcl", "csubj"}
    parts = label.split(":")
    return label if parts[0] in keep and len(parts) > 1 else parts[0]


def build_ud_stanza(text: str, max_depth: int = 2) -> str:
    """
    Парсит русский текст через Stanza (ru_syntagrus).

    Возвращает строку вида:
      ROOT[UPOS](лемма) + deprel(форма/UPOS) + ...
    Вложенные зависимые (глубина ≤ max_depth):
      deprel(форма/UPOS > [inner_dep(форма/UPOS), ...])
    """
    if not text or not str(text).strip():
        return ""
    text = str(text).strip().rstrip(".")
    doc = nlp_ru(text)
    if not doc.sentences:
        return text

    words = doc.sentences[0].words
    children = {w.id: [] for w in words}
    root_word = None

    for w in words:
        if w.deprel and w.deprel.lower() == "root":
            root_word = w
        if w.head and w.head in children and w.head != w.id:
            children[w.head].append(w)

    if root_word is None:
        root_word = words[0]

    def subtree(w, depth=0):
        if (w.upos or "X") == "PUNCT":
            return None
        kids = [c for c in sorted(children.get(w.id, []), key=lambda x: x.id)
                if (c.upos or "X") != "PUNCT"]
        parts = []
        for kid in kids:
            rel = _fmt_deprel(kid.deprel or "dep")
            grandkids = [gc for gc in sorted(children.get(kid.id, []), key=lambda x: x.id)
                         if (gc.upos or "X") != "PUNCT"]
            if grandkids and depth < max_depth:
                inner = ", ".join(f"{_fmt_deprel(gc.deprel or 'dep')}({gc.text}/{gc.upos})"
                                  for gc in grandkids)
                parts.append(f"{rel}({kid.text}/{kid.upos} > [{inner}])")
            else:
                parts.append(f"{rel}({kid.text}/{kid.upos})")
        label = "ROOT" if w == root_word else _fmt_deprel(w.deprel or "dep").upper()
        base = f"{label}[{w.upos or 'X'}]({w.lemma or w.text})"
        return (base + " + " + " + ".join(parts)) if parts else base

    return subtree(root_word) or text


def build_ud_spacy(text: str, max_depth: int = 2) -> str:
    """
    Парсит английский текст через spaCy (en_core_web_trf).
    Нотация идентична build_ud_stanza.
    """
    if not text or not str(text).strip():
        return ""
    text = str(text).strip().rstrip(".")
    doc = nlp_en(text)

    children = {t.i: [] for t in doc}
    root_tok = None
    for t in doc:
        if t.dep_ == "ROOT":
            root_tok = t
        if t.dep_ != "ROOT" and t.head.i != t.i:
            children[t.head.i].append(t)
    if root_tok is None:
        root_tok = doc[0]

    def subtree(t, depth=0):
        if t.pos_ in ("PUNCT", "SPACE"):
            return None
        kids = [c for c in sorted(children.get(t.i, []), key=lambda x: x.i)
                if c.pos_ not in ("PUNCT", "SPACE")]
        parts = []
        for kid in kids:
            rel = _fmt_deprel(kid.dep_)
            grandkids = [gc for gc in sorted(children.get(kid.i, []), key=lambda x: x.i)
                         if gc.pos_ not in ("PUNCT", "SPACE")]
            if grandkids and depth < max_depth:
                inner = ", ".join(f"{_fmt_deprel(gc.dep_)}({gc.text}/{gc.pos_})"
                                  for gc in grandkids)
                parts.append(f"{rel}({kid.text}/{kid.pos_} > [{inner}])")
            else:
                parts.append(f"{rel}({kid.text}/{kid.pos_})")
        label = "ROOT" if t == root_tok else _fmt_deprel(t.dep_).upper()
        base = f"{label}[{t.pos_}]({t.lemma_})"
        return (base + " + " + " + ".join(parts)) if parts else base

    return subtree(root_tok) or text


# ── Тест на примерах из корпуса ──────────────────────────────────────────
tests = [
    ("RU", "запах гари"),
    ("RU", "невыносимый смрад"),
    ("RU", "грузовик дохнул раскаленной вонью"),
    ("RU", "не вытравил из себя сладко-смердящего матушкина духа"),
    ("RU", "зефир, шумящий древесами, веет нам благоуханием, собранным со цветов"),
    ("EN", "smell of burning"),
    ("EN", "unbearable stench"),
    ("EN", "truck breathed scorching stench"),
    ("EN", "not purged the sweetly festering spirit"),
    ("EN", "zephyr wafts fragrance gathered from the flowers"),
]
print("=" * 65)
for lang, phrase in tests:
    fn = build_ud_stanza if lang == "RU" else build_ud_spacy
    print(f"\n[{lang}] «{phrase}»")
    print(f"  → {fn(phrase)}")



[RU] «запах гари»
  → ROOT[NOUN](запах) + nmod(гари/NOUN)

[RU] «невыносимый смрад»
  → ROOT[NOUN](смрад) + amod(невыносимый/ADJ)

[RU] «грузовик дохнул раскаленной вонью»
  → ROOT[VERB](дохнуть) + nsubj(грузовик/NOUN) + obl(вонью/NOUN > [amod(раскаленной/VERB)])

[RU] «не вытравил из себя сладко-смердящего матушкина духа»
  → ROOT[VERB](вытравить) + advmod(не/PART) + obl(себя/PRON > [case(из/ADP)]) + nsubj(матушкина/NOUN > [acl(смердящего/VERB), nmod(духа/NOUN)])

[RU] «зефир, шумящий древесами, веет нам благоуханием, собранным со цветов»
  → ROOT[VERB](веять) + nsubj(зефир/NOUN > [acl(шумящий/VERB)]) + iobj(нам/PRON) + obl(благоуханием/NOUN > [acl(собранным/VERB)])

[EN] «smell of burning»
  → ROOT[NOUN](smell) + prep(of/ADP > [pobj(burning/NOUN)])

[EN] «unbearable stench»
  → ROOT[NOUN](stench) + amod(unbearable/ADJ)

[EN] «truck breathed scorching stench»
  → ROOT[VERB](breathe) + nsubj(truck/NOUN) + dobj(stench/NOUN > [amod(scorching/VERB)])

[EN] «not purged the sweetly festeri

## ЯЧЕЙКА 4 — Загрузка файла, парсинг, запись результатов ##

In [ ]:
print("Загрузи файл Разметка_сравнение_RU_EN.xlsx:")
uploaded = files.upload()
FILENAME = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(FILENAME)
ws = wb["Разметка"]

headers = {ws.cell(2, c).value: c for c in range(1, ws.max_column + 1)}
COL_CONCEPT_RU = headers["concept_unit_RU"]
COL_GRAM_RU    = headers["gram_structure"]
COL_CONCEPT_EN = headers["concept_unit_EN"]
COL_GRAM_EN    = headers["gram.structure_EN"]

print(f"concept_unit_RU → col {COL_CONCEPT_RU} | gram_structure → col {COL_GRAM_RU}")
print(f"concept_unit_EN → col {COL_CONCEPT_EN} | gram.structure_EN → col {COL_GRAM_EN}")

# Цвета в соответствии с существующей схемой таблицы
C_RU_LIGHT = "D6E4F0"; C_ALT_RU = "EBF5FB"
C_EN_LIGHT = "D5F5E3"; C_ALT_EN = "EAFAF1"

def _fill(h): return PatternFill("solid", start_color=h, fgColor=h)
def _border():
    s = Side(style="thin", color="AAAAAA")
    return Border(left=s, right=s, top=s, bottom=s)

def write_cell(ws, row, col, value, bg):
    c = ws.cell(row=row, column=col)
    c.value = value
    c.font = Font(name="Arial", size=9)
    c.fill = _fill(bg)
    c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    c.border = _border()

done_ru = done_en = 0
errors_ru, errors_en = [], []

print(f"\nПарсинг {ws.max_row - 2} строк...")

for row in range(3, ws.max_row + 1):
    even = ((row - 3) % 2 == 0)
    ru_bg = C_ALT_RU if even else C_RU_LIGHT
    en_bg = C_ALT_EN if even else C_EN_LIGHT

    # RU — Stanza
    c_ru = ws.cell(row, COL_CONCEPT_RU).value
    if c_ru and str(c_ru).strip() not in ("", "None"):
        try:
            write_cell(ws, row, COL_GRAM_RU, build_ud_stanza(str(c_ru)), ru_bg)
            done_ru += 1
        except Exception as e:
            write_cell(ws, row, COL_GRAM_RU, f"[ERROR: {e}]", ru_bg)
            errors_ru.append((row, str(c_ru)[:40], str(e)))

    # EN — spaCy
    c_en = ws.cell(row, COL_CONCEPT_EN).value
    if c_en and str(c_en).strip() not in ("", "None"):
        try:
            write_cell(ws, row, COL_GRAM_EN, build_ud_spacy(str(c_en)), en_bg)
            done_en += 1
        except Exception as e:
            write_cell(ws, row, COL_GRAM_EN, f"[ERROR: {e}]", en_bg)
            errors_en.append((row, str(c_en)[:40], str(e)))

    if row % 25 == 0:
        print(f"  строка {row}/{ws.max_row}")

print(f"\n✓ RU разобрано: {done_ru} | EN разобрано: {done_en}")
if errors_ru: print(f"⚠ Ошибки RU: {[(r,t) for r,t,_ in errors_ru]}")
if errors_en: print(f"⚠ Ошибки EN: {[(r,t) for r,t,_ in errors_en]}")

Загрузи файл Разметка_сравнение_RU_EN.xlsx:


KeyboardInterrupt: 

## ЯЧЕЙКА 5 — Сохранение и скачивание ##

In [ ]:
OUTPUT = "Разметка_сравнение_RU_EN_parsed.xlsx"
wb.save(OUTPUT)
print(f"✓ Сохранено: {OUTPUT}")
files.download(OUTPUT)


✓ Сохранено: Разметка_сравнение_RU_EN_parsed.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# ОТЧЁТ пот тестовым данным


# Токен → грамматическая структура: корреляции в ольфакторном корпусе RU/EN

**Корпус:** 66 пар concept\_unit (RU + EN-перевод), 14 RU-токенов, 16 EN-токенов.  
**Парсеры:** Stanza ru\_syntagrus (RU) + spaCy en\_core\_web\_trf (EN).  
**Нотация:** `ROOT[UPOS](лемма) + deprel(форма/UPOS > [вложенные зависимые])`.

---

## 1. Главный результат

Синтаксическая роль (HEAD / DEP) ольфакторного токена **сохраняется при переводе в 91% случаев** (60/66). Это верно для всех типов номинации:

| type\_RU | Совпадение HEAD/DEP RU↔EN |
|-----------|--------------------------|
| direct    | 36/40 — 90%              |
| metaphor  | 19/21 — 90%              |
| idiom     | 3/3 — 100%               |
| synesth   | 2/2 — 100%               |

Переводчик воспроизводит не только лексический эквивалент, но и **позицию слова в синтаксическом дереве**.

---

## 2. Синтаксические профили токенов

Из данных выводятся четыре устойчивых профиля:

| Профиль | RU-токены | EN-токены |
|---------|-----------|-----------|
| **NP-якорь** | запах, смр ад | smell, scent |
| **Квазиагент** | вонь, аромат | stench, fragrance, aroma |
| **Предикат** | вонять, смердеть, пахнуть | reek (VERB), smelled |
| **Атрибут** | смердящий, источающий | reeking, stinking, festering |

---

## 3. Русский: разбор по токенам

### запах (n = 15) — NP-якорь, HEAD 93%

Доминирующий паттерн — в 14 из 15 случаев:

```
ROOT[NOUN](запах) + nmod(X/NOUN)
ROOT[NOUN](запах) + nmod(X/NOUN > [amod(ADJ)])
```

Запах — синтаксически самый стабильный токен корпуса. Всегда вершина именной группы, генитивный nmod обязателен, amod факультативен. Ни одного предикативного употребления.

Единственное отклонение — «запах курицы-гриль»:

```
ROOT[NOUN](гриль) + nsubj(запах/NOUN > [nmod(курицы/NOUN)])
```

Парсер поставил *гриль* в root из-за дефиса — артефакт токенизации, а не синтаксический сдвиг.

---

### смр ад (n = 5) — NP-якорь, HEAD 80%

```
ROOT[NOUN](смр ад) + amod(ADJ)
ROOT[NOUN](смр ад) + amod(ADJ) + nmod(X/NOUN)
ROOT[NOUN](смр ад) + nmod(X/NOUN)
```

Структурно близок к *запаху*, но с принципиальным различием: **amod появляется чаще** — в 4 из 5 против 4 из 14 у *запаха*. Смр ад «требует» оценочного прилагательного (*невыносимый, слабый, жуткий*). Это отражает семантику: смр ад уже несёт отрицательную оценку, но она регулярно усиливается атрибутом.

Одно отклонение — координативное перечисление:

```
ROOT[NOUN](клопа) + conj(смр ад/NOUN) + conj(сырость/NOUN)
```

Смр ад теряет вершинную позицию в пользу другого существительного.

---

### аромат (n = 11) — расщеплённый профиль, HEAD 45%

Два равновероятных паттерна:

**Паттерн A — NP-вершина (5/11):**

```
ROOT[NOUN](аромат) + amod(ADJ) + nmod(X/NOUN)
ROOT[NOUN](аромат) + nmod(X/NOUN)
```

**Паттерн B — nsubj при активном глаголе (4/11):**

```
ROOT[VERB](распространяться) + nsubj(аромат/NOUN > [nmod(...)])
ROOT[VERB](мочь)             + nsubj(аромат/NOUN)
ROOT[VERB](исходить)         + nsubj(аромат/NOUN > [amod(...)])
```

Это принципиальное отличие от *запаха*: **аромат регулярно выступает агентом действия**. Глаголы при аромат-субъекте — *распространяться, исходить, мочь* — либо движение в пространстве, либо потенциальность воздействия. Аромат «делает» что-то с воспринимающим.

---

### вонь (n = 8) — квазиагент, HEAD 0%

Вонь **ни разу не занимает вершину** концептуальной единицы. Две конструкции:

**nsubj при агентивном глаголе (5/8):**

```
ROOT[VERB](терзать)          + nsubj(вонь/NOUN > [amod(жуткая)])
ROOT[VERB](распространяться) + nsubj(вонь/NOUN)
ROOT[VERB](подняться)        + nsubj(вонь/NOUN)
ROOT[VERB](дохнуть)          + nsubj(грузовик/NOUN) + obl(вонью/NOUN)
```

**obl / nmod при других вершинах (3/8):**

```
ROOT[VERB](пахнуть)    + xcomp(прогоркло/VERB > [obl(вонью)])
ROOT[VERB](пропитаться) + obl(вонью/NOUN > [amod(...)])
ROOT[ADV](много)        + nsubj(вони/NOUN) + cop(будет)
```

Глаголы при вонь-субъекте — *терзать, подняться, дохнуть* — физически активные, с семантикой принудительного воздействия. **Интенсивный неприятный запах концептуализируется как агент, действующий на воспринимающего**.

---

### вонять (n = 6) — предикат, HEAD 100%

Самый синтаксически однородный глагольный токен. Всегда ROOT[VERB](вонять):

```
ROOT[VERB](вонять) + nsubj(X/NOUN)
ROOT[VERB](вонять) + nsubj(X/NOUN) + obl(Y/NOUN)
ROOT[VERB](вонять) + advmod(густо/ADV) + nsubj(X/NOUN)
ROOT[VERB](вонять) + nsubj(X) + aux(будет) + ccomp(...)
```

Субъекты при *вонять*: машины, люди, шелуха, грибная сырость — конкретные предметы и существа. Вонять не метафоризируется: это глагол физической, перцептивной реальности.

---

### смердеть (n = 4) — предикат, HEAD 75%

В большинстве случаев ROOT[VERB] с субъектом:

```
ROOT[VERB](смердить) + nsubj(ты/PRON > [advmod(же)]) + advmod(уже)
ROOT[VERB](смердеть) + nsubj(мозг/NOUN) + obl(пленым/NOUN)
ROOT[VERB](смердеть) + nsubj(Труб/NOUN > [nmod(самодержавия)]) + advmod(ещё)
```

Субъекты *смердеть* принципиально иные, чем у *вонять*: мозг, труба самодержавия, «ты» — **метафорические, абстрактные или социально нагруженные референты**. Это соответствует книжному, церковнославянскому регистру токена.

---

### смердящий (n = 5) — атрибут, HEAD 0%

Никогда не вершина. Две синтаксические ниши:

**acl при существительном (3/5):**

```
ROOT[NOUN](воздух) + acl(смердящего/VERB > [obl(останками)])
ROOT[NOUN](псы)    + amod(смердящие/VERB)
ROOT[NOUN](матушкина) + acl(смердящего/VERB > [...])
```

**amod в составе клаузы с глагольным root (2/5):**

```
ROOT[VERB](завалить)    + obl(тряпьем > [conj(овчинами)])
ROOT[VERB](открываться) + nsubj(потоки > [nmod(гноя)])
```

Причастие фиксирует **постоянный признак носителя**, а не отдельное действие. Синтаксическая зависимость — прямое отражение семантической вторичности атрибута.

---

### источать / источающий (n = 2 + 2) — глагол vs. причастие

**источать** (VERB, HEAD 100%):

```
ROOT[VERB](источать) + obj(обаяние/NOUN > [amod(скромное), nmod(буржуазии)])
ROOT[VERB](источать) + nsubj(нефть) + mark(словно) + obj(га/NOUN > [amod(веселящий)])
```

**источающий** (PRTCP, HEAD 0%):

```
ROOT[NOUN](запах)  + amod(источавшая/VERB) + amod(густой/ADJ) + nmod(колбасы)
ROOT[NOUN](аромат) + amod(источающих/VERB) + amod(медвяный/ADJ) + nmod(лип)
```

Зеркальная пара: глагольная форма — вершина с объектом, причастная — атрибут при существительном-вершине. Грамматическая форма **полностью предопределяет** синтаксическую позицию.

---

## 4. Английский: разбор по токенам

### smell (n = 12) — NP-якорь, HEAD 100%

Абсолютно однородный паттерн:

```
ROOT[NOUN](smell) + prep(of/ADP > [pobj(X/NOUN)])
ROOT[NOUN](smell) + amod(ADJ) + prep(of/ADP > [pobj(X/NOUN)])
```

11 из 12 случаев — одна конструкция. Точный структурный эквивалент *запаха*: NP-вершина с обязательным предложным дополнением `of + NOUN`. Предлог *of* функционирует как аналог русского родительного падежа.

Одно отклонение — smelling как причастие в acl:

```
ROOT[NOUN](air) + acl(reeking/VERB > [prep(of)]) + acl(smelling/VERB > [advmod(faintly), prep(of)])
```

Финитная форма *smelled* демонстрирует **грамматическую омонимию** smell-NOUN и smell-VERB: при переходе в предикат структура полностью меняется:

```
ROOT[VERB](smell) + nsubj(smoke/NOUN) + advmod(rancidly/ADV)
```

---

### scent (n = 3) — NP-якорь, HEAD 100%

```
ROOT[NOUN](scent) + prep(of/ADP > [pobj(X/NOUN)])
```

Три из трёх — одна структура. Идентичен *smell* по синтаксическому профилю. Семантическое различие (scent = природный/тонкий запах) не отражается в синтаксисе.

---

### stench (n = 11) — расщеплённый профиль, HEAD 36%

Самый вариативный EN-токен. Три ниши:

**HEAD — NP с amod (4/11):**

```
ROOT[NOUN](stench) + amod(unbearable/ADJ)
ROOT[NOUN](stench) + amod(faint/ADJ) + prep(of/ADP > [pobj(shoe)])
ROOT[NOUN](stench) + prep(of/ADP > [pobj(luxury)])
```

**nsubj при активном глаголе (5/11):**

```
ROOT[VERB](torment) + nsubj(stench/NOUN > [amod(dreadful)])
ROOT[VERB](spread)  + nsubj(stench/NOUN)
ROOT[VERB](rise)    + nsubj(stench/NOUN > [amod(terrible)])
ROOT[VERB](breathe) + nsubj(truck/NOUN) + dobj(stench/NOUN)
```

**pobj при глаголе (2/11):**

```
ROOT[VERB](saturate) + prep(with/ADP > [pobj(stench)])
ROOT[VERB](smell)    + prep(of/ADP > [pobj(stench)])
```

Stench — прямой структурный эквивалент *вони*: оба регулярно выступают субъектом при активных глаголах с семантикой воздействия (*torment, rise, spread* ≈ *терзать, подняться, распространяться*). **Квазиагентивная конструкция сохраняется в переводе**.

---

### reek (n = 10) — предикат, HEAD 77%

Преобладающий паттерн — VERB-вершина с субъектом:

```
ROOT[VERB](reek) + nsubj(X/NOUN) + prep(of/ADP > [pobj(Y/NOUN)])
ROOT[VERB](reek) + nsubj(X/NOUN) + advmod(ADV)
ROOT[VERB](reek) + nsubj(X) + aux(are) + advmod(already) + prep(of/ADP)
```

Как существительное — только 2 случая:

```
ROOT[NOUN](torment) + prep(of/ADP > [pobj(reek/NOUN)])
ROOT[NOUN](reek)    + prep(of/ADP > [pobj(tobacco/NOUN)])
```

*Reek* — **по умолчанию глагол**, в отличие от *stench*, который по умолчанию существительное. Оба обозначают интенсивный неприятный запах, но тяготеют к разным частеречным нишам. Субъекты при *reek*: people, shells, brain, liver, pipe, air — диапазон шире, чем у русского *смердеть*.

---

### stink (n = 4) — высокая вариативность, HEAD 25%

```
ROOT[VERB](stink)  + nsubj(cars) + prep(with > [pobj(fumes)])
ROOT[VERB](die)    + prep(from/ADP > [pobj(stink/NOUN)])
ROOT[AUX](be)      + attr(lot/NOUN > [prep(of/ADP > [pobj(stink)])])
ROOT[VERB](kick)   + dobj(stink/NOUN) + prep(about/ADP > [...])
```

Наибольшая синтаксическая полисемия среди EN-токенов. *Kick up a stink* — фразеологизм, где *stink* десемантизирован до «скандал/переполох». Именно эта единица при переводе меняет тип номинации: RU `metaphor` → EN `idiom` — один из шести случаев несовпадения ролей в корпусе.

---

### aroma (n = 6) — расщеплённый профиль, HEAD 50%

**HEAD — NP (3/6):**

```
ROOT[NOUN](aroma) + prep(of/ADP > [pobj(X)])
ROOT[NOUN](aroma) + amod(ADJ) + amod(ADJ) + prep(of/ADP > [...])
```

**DEP при глаголе (3/6):**

```
ROOT[VERB](touch) + agent(by/ADP > [pobj(aroma)])
ROOT[VERB](drift) + nsubj(aroma/NOUN > [amod(perceptible)])
ROOT[VERB](give)  + dobj(aroma/NOUN > [amod(sharp)])
```

Как и русский *аромат*, EN *aroma* раздвоен между именной группой и субъектом / объектом при глаголе. **Профиль сохраняется в переводе**.

---

### fragrance (n = 7) — квазиагент, HEAD 43%

```
ROOT[VERB](lull)   + nsubj(fragrance/PROPN) + aux(could)
ROOT[VERB](spread) + nsubj(fragrance/NOUN > [prep(of)])
ROOT[VERB](waft)   + nsubj(zephyr) + dobj(fragrance/NOUN > [acl(gathered)])
```

*Fragrance* — самый агентивный позитивный токен: субъект при *lull, spread, waft* — глаголах с семантикой намеренного воздействия. Это структурное зеркало *stench/вони* в позитивном полюсе: те же нагруженные глаголы, но с другим знаком.

---

### Атрибутивные причастия: reeking / stinking / festering

```
ROOT[VERB](heap)  + prep(with > [pobj(rags)])          ← reeking sheepskins: amod в pobj
ROOT[NOUN](dog)   + amod(stinking/VERB)
ROOT[VERB](purge) + dobj(spirit > [amod(festering)])
```

Все три — DEP, все три — amod/acl при существительном. Прямой структурный аналог *смердящего* и *источающего*. Причастный атрибут занимает одну и ту же нишу в RU и EN.

---

## 5. Сводная таблица

| Токен RU | Токен EN | HEAD% RU | HEAD% EN | Root UPOS RU | Root UPOS EN | Профиль |
|---|---|---|---|---|---|---|
| запах | smell / scent | 93% | 100% | NOUN | NOUN | NP-якорь |
| смр ад | stench | 80% | 36% | NOUN | VERB / NOUN | NP → квазиагент |
| аромат | aroma / fragrance | 45% | 47% | NOUN / VERB | NOUN / VERB | расщеплённый |
| вонь | stench | 0% | 36% | VERB | VERB / NOUN | квазиагент |
| вонять | stink (VERB) | 100% | 25% | VERB | VERB | предикат |
| смердеть | reek (VERB) | 75% | 77% | VERB | VERB | предикат |
| смердящий | reeking / stinking | 0% | 0% | NOUN / VERB | VERB / NOUN | атрибут |
| источающий | exuding | 0% | 0% | NOUN | NOUN | атрибут |
| источать | exudes | 100% | 100% | VERB | VERB | предикат |

---

## 6. Ключевые наблюдения

**1. Структурная устойчивость нейтрального обозначения.** *Запах* и *smell/scent* образуют единственную по-настоящему жёсткую конструкцию корпуса: HEAD NOUN + genitive/of-NP, без вариантов. Нейтральная лексика синтаксически стабильнее экспрессивной.

**2. Квазиагентивность интенсивного отрицательного запаха.** *Вонь* (0% HEAD) и *stench* (36% HEAD) регулярно появляются как nsubj при глаголах физического и аффективного воздействия — *терзать, подняться, torment, rise, spread*. Это устойчивая **концептуальная метафора**: интенсивный неприятный запах = агент, принудительно действующий на воспринимающего. *Запах* и *smell* так не ведут себя никогда.

**3. Расщепление аромат / fragrance.** Оба делятся поровну между именной вершиной и субъектом действия. Позитивный интенсивный запах тоже получает агентивную конструкцию — но глаголы другие: *усыпить, распространяться, lull, waft* — с семантикой мягкого, желанного воздействия.

**4. Причастная атрибутивность как типологическая универсалия.** *Смердящий, источающий, reeking, stinking, festering* — все 0% HEAD, все amod/acl. Причастный атрибут не является предикатом ни в одном языке. Это прямое отражение семантики: признак, а не действие.

**5. Глагольная / именная специализация в EN при одном денотате.** *Reek* тяготеет к VERB (77% HEAD, ROOT[VERB]), *stench* — к NOUN и квазиагентивным конструкциям. В русском это разведено лексически (*смердеть* vs. *вонь*); в английском выражено двумя леммами с разными частеречными профилями внутри одного семантического поля.

**6. Синтаксический сдвиг при переводе — маркер фразеологизации.** Все 6 несовпадений HEAD/DEP (из 66) приходятся на случаи, где переводчик заменил живую метафору идиомой (*kick up a stink*) или перестроил клаузу. Синтаксическая роль меняется именно там, где меняется коммуникативная функция высказывания, а не просто лексема.




# ____ЭТАП II____

## ЯЧЕЙКА 1 — Установка библиотек ##

# Новый раздел

In [ ]:
!pip install -q sentence-transformers openpyxl scipy

print("✓ Библиотеки установлены")


✓ Библиотеки установлены


## ЯЧЕЙКА 2 — Импорты и загрузка модели LaBSE ##

In [ ]:
import torch
import numpy as np
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from sentence_transformers import SentenceTransformer
from scipy.stats import chi2_contingency, fisher_exact
from scipy.spatial.distance import cosine
from collections import Counter, defaultdict
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("Загрузка LaBSE (~1–2 мин)...")
# LaBSE (Language-agnostic BERT Sentence Embeddings) — оптимальна для
# межъязыкового сравнения: обучена на 109 языках, включая RU и EN,
# специально для cross-lingual semantic similarity
model = SentenceTransformer('sentence-transformers/LaBSE')
print(f"✓ LaBSE загружена | device: {model.device}")

Загрузка LaBSE (~1–2 мин)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ LaBSE загружена | device: cpu


## ЯЧЕЙКА 3 — Загрузка файла и извлечение данных ##

In [ ]:
print("Загрузи файл Разметка_сравнение_RU_EN_parsed.xlsx:")
uploaded = files.upload()
FILENAME = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(FILENAME)
ws = wb['Разметка']

# Карта заголовков
headers = {ws.cell(2, c).value: c for c in range(1, ws.max_column + 1)}

# Колонки
C_CU_RU    = headers['concept_unit_RU']    # G = 7
C_CU_EN    = headers['concept_unit_EN']    # N = 14
C_TYPE_RU  = headers['type_RU']            # H = 8
C_TYPE_EN  = headers['type_EN']            # R = 18
C_CONN_RU  = headers['connotation_RU']     # I = 9
C_CONN_EN  = headers['connotation_EN']     # S = 19
C_TONE_RU  = headers['тональность_RU']     # J = 10
C_TONE_EN  = headers['тональность_EN']     # T = 20
C_SHIFT    = headers['translation_shift']  # V = 22
C_COS      = headers['cosine_sim_LaBSE']   # W = 23
C_NOTES    = headers['shift_notes']        # X = 24
C_VERIFIED = headers['verified']           # Y = 25

print(f"\nКолонки:")
print(f"  concept_unit_RU  → col {C_CU_RU}")
print(f"  concept_unit_EN  → col {C_CU_EN}")
print(f"  translation_shift → col {C_SHIFT}")
print(f"  cosine_sim_LaBSE  → col {C_COS}")
print(f"  shift_notes       → col {C_NOTES}")
print(f"  verified          → col {C_VERIFIED}")

# Собираем данные построчно
rows = []
for r in range(3, ws.max_row + 1):
    cu_ru   = ws.cell(r, C_CU_RU).value
    cu_en   = ws.cell(r, C_CU_EN).value
    type_ru = ws.cell(r, C_TYPE_RU).value
    type_en = ws.cell(r, C_TYPE_EN).value
    conn_ru = ws.cell(r, C_CONN_RU).value
    conn_en = ws.cell(r, C_CONN_EN).value
    tone_ru = ws.cell(r, C_TONE_RU).value
    tone_en = ws.cell(r, C_TONE_EN).value
    rows.append({
        'row': r,
        'cu_ru': cu_ru, 'cu_en': cu_en,
        'type_ru': type_ru, 'type_en': type_en,
        'conn_ru': conn_ru, 'conn_en': conn_en,
        'tone_ru': tone_ru, 'tone_en': tone_en,
    })

# Только строки с обеими концептуальными единицами
paired = [d for d in rows if d['cu_ru'] and d['cu_en']
          and str(d['cu_ru']).strip() not in ('', 'None')
          and str(d['cu_en']).strip() not in ('', 'None')]

print(f"\nСтрок с парами RU+EN: {len(paired)}")
print(f"Всего строк: {len(rows)}")

Загрузи файл Разметка_сравнение_RU_EN_parsed.xlsx:


Saving Разметка_сравнение_RU_EN_parsed (2604091724).xlsx to Разметка_сравнение_RU_EN_parsed (2604091724).xlsx

Колонки:
  concept_unit_RU  → col 7
  concept_unit_EN  → col 14
  translation_shift → col 22
  cosine_sim_LaBSE  → col 23
  shift_notes       → col 24
  verified          → col 25

Строк с парами RU+EN: 172
Всего строк: 185


## ЯЧЕЙКА 4 — Вычисление LaBSE косинусного сходства ##

In [ ]:
print("Кодирование concept_unit_RU...")
texts_ru = [str(d['cu_ru']).strip() for d in paired]
texts_en = [str(d['cu_en']).strip() for d in paired]

# Батч-кодирование — эффективнее одиночного
emb_ru = model.encode(texts_ru, batch_size=32, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True)
emb_en = model.encode(texts_en, batch_size=32, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True)

# Косинусное сходство: при normalize_embeddings=True → dot product = cosine similarity
cosine_sims = (emb_ru * emb_en).sum(axis=1)

# Привязываем к данным
for i, d in enumerate(paired):
    d['cosine_sim'] = float(cosine_sims[i])

# Статистика
sims = [d['cosine_sim'] for d in paired]
print(f"\n{'='*50}")
print(f"LaBSE cosine similarity — статистика:")
print(f"  n        = {len(sims)}")
print(f"  mean     = {np.mean(sims):.4f}")
print(f"  median   = {np.median(sims):.4f}")
print(f"  std      = {np.std(sims):.4f}")
print(f"  min      = {np.min(sims):.4f}")
print(f"  max      = {np.max(sims):.4f}")
print(f"  ≥0.90    = {sum(1 for s in sims if s >= 0.90)} ({sum(1 for s in sims if s >= 0.90)/len(sims)*100:.1f}%)")
print(f"  0.75–0.89 = {sum(1 for s in sims if 0.75 <= s < 0.90)} ({sum(1 for s in sims if 0.75 <= s < 0.90)/len(sims)*100:.1f}%)")
print(f"  <0.75    = {sum(1 for s in sims if s < 0.75)} ({sum(1 for s in sims if s < 0.75)/len(sims)*100:.1f}%)")


Кодирование concept_unit_RU...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]


LaBSE cosine similarity — статистика:
  n        = 172
  mean     = 0.7372
  median   = 0.7618
  std      = 0.1513
  min      = 0.1206
  max      = 0.9735
  ≥0.90    = 25 (14.5%)
  0.75–0.89 = 66 (38.4%)
  <0.75    = 81 (47.1%)


## ЯЧЕЙКА 5 — Разметка translation_shift и shift_notes ##

In [ ]:
# ── Пороги и логика классификации ────────────────────────────────────────
THRESHOLD_EQUIV   = 0.90   # ≥0.90 → equivalent по умолчанию
THRESHOLD_SHIFT   = 0.75   # 0.75–0.89 → shift
# <0.75 → significant_shift

def classify_shift(d):
    """
    Мультимерная классификация переводческого сдвига.
    Учитывает: cosine_sim + type + connotation + тональность + POS токена.

    Возвращает (shift_label, notes, needs_review).

    Логика:
      1. Если cos >= EQUIV И все категории совпадают → equivalent
      2. Если cos >= EQUIV НО есть категориальный сдвиг → lex (лексич. замена при семант. близости)
      3. Если SHIFT <= cos < EQUIV → shift (умеренный сдвиг)
      4. Если cos < SHIFT → significant_shift (значительный сдвиг)

    Дополнительные метки через "+":
      +type_shift   — type_RU ≠ type_EN
      +tone_shift   — тональность_RU ≠ тональность_EN
      +conn_shift   — connotation_RU ≠ connotation_EN
    """
    cos    = d['cosine_sim']
    t_ru   = d.get('type_ru') or ''
    t_en   = d.get('type_en') or ''
    c_ru   = d.get('conn_ru') or ''
    c_en   = d.get('conn_en') or ''
    tn_ru  = d.get('tone_ru') or ''
    tn_en  = d.get('tone_en') or ''

    type_match = (t_ru == t_en)
    conn_match = (c_ru == c_en)
    tone_match = (tn_ru == tn_en)
    all_match  = type_match and conn_match and tone_match

    # Базовая категория по косинусу
    if cos >= THRESHOLD_EQUIV:
        base = "equivalent" if all_match else "lex"
    elif cos >= THRESHOLD_SHIFT:
        base = "shift"
    else:
        base = "significant_shift"

    # Дополнительные метки
    extras = []
    if not type_match:
        extras.append(f"type_shift({t_ru}→{t_en})")
    if not conn_match:
        extras.append(f"conn_shift({c_ru}→{c_en})")
    if not tone_match:
        extras.append(f"tone_shift({tn_ru}→{tn_en})")

    label = base + (("+" + "+".join(extras)) if extras else "")

    # Заметки
    note_parts = [f"cos={cos:.3f}"]
    if not type_match:
        note_parts.append(f"type:{t_ru}→{t_en}")
    if not conn_match:
        note_parts.append(f"conn:{c_ru}→{c_en}")
    if not tone_match:
        note_parts.append(f"tone:{tn_ru}→{tn_en}")
    if not note_parts[1:]:
        note_parts.append("все категории совпадают")

    notes = " | ".join(note_parts)

    # Флаг для ручной проверки: крайние значения или противоречия
    needs_review = (
        cos < 0.70               # очень низкое сходство
        or cos > 0.97            # подозрительно высокое — возможна копия
        or (cos >= THRESHOLD_EQUIV and not all_match)  # высокий cos, но категориальный сдвиг
        or (base == "significant_shift")
    )

    return label, notes, needs_review


# Применяем
for d in paired:
    d['shift_label'], d['shift_notes'], d['needs_review'] = classify_shift(d)

# Превью
print("=" * 65)
print("PREVIEW — первые 15 строк:")
print(f"{'row':>4} {'cos':>6} {'shift':<35} {'review':>7}")
print("-" * 65)
for d in paired[:15]:
    rv = "⚑ YES" if d['needs_review'] else "no"
    print(f"R{d['row']:>3} {d['cosine_sim']:>6.3f} {d['shift_label']:<35} {rv:>7}")

# Крайние случаи
print("\n── Низкое сходство (cos < 0.75) ──")
low = [d for d in paired if d['cosine_sim'] < 0.75]
for d in sorted(low, key=lambda x: x['cosine_sim']):
    print(f"  R{d['row']:>3} cos={d['cosine_sim']:.3f}  RU: {str(d['cu_ru'])[:35]}  →  EN: {str(d['cu_en'])[:35]}")

print("\n── Высокое сходство (cos > 0.90) ──")
high = [d for d in paired if d['cosine_sim'] > 0.90]
for d in sorted(high, key=lambda x: -x['cosine_sim']):
    print(f"  R{d['row']:>3} cos={d['cosine_sim']:.3f}  RU: {str(d['cu_ru'])[:35]}  →  EN: {str(d['cu_en'])[:35]}")


PREVIEW — первые 15 строк:
 row    cos shift                                review
-----------------------------------------------------------------
R  3  0.810 shift                                    no
R  4  0.645 significant_shift                     ⚑ YES
R  5  0.695 significant_shift                     ⚑ YES
R  6  0.819 shift                                    no
R  7  0.936 equivalent                               no
R  8  0.742 significant_shift                     ⚑ YES
R  9  0.683 significant_shift                     ⚑ YES
R 10  0.674 significant_shift                     ⚑ YES
R 11  0.692 significant_shift                     ⚑ YES
R 12  0.635 significant_shift                     ⚑ YES
R 13  0.714 significant_shift                     ⚑ YES
R 14  0.630 significant_shift                     ⚑ YES
R 15  0.932 equivalent                               no
R 16  0.859 shift                                    no
R 18  0.867 shift                                    no

── Низкое 

## ЯЧЕЙКА 6 — Статистический анализ: χ² и Fisher's exact ##

In [ ]:
def contingency_analysis(col_ru, col_en, paired_data, label):
    """
    Строит таблицу сопряжённости, вычисляет χ² и, если малые ожидаемые частоты, Fisher.
    Работает с двумя распределениями:
      a) Paired: RU vs EN для тех же 66 строк (McNemar / paired χ²)
      b) Marginal: общее распределение RU (66) vs EN (66)
    """
    print(f"\n{'='*60}")
    print(f"  {label}")
    print('='*60)

    vals_ru = [d.get(col_ru) for d in paired_data if d.get(col_ru)]
    vals_en = [d.get(col_en) for d in paired_data if d.get(col_en)]

    cnt_ru = Counter(vals_ru)
    cnt_en = Counter(vals_en)

    # Все категории
    cats = sorted(set(list(cnt_ru.keys()) + list(cnt_en.keys())))

    print(f"\n  Категория          RU     EN    Δ(EN-RU)")
    print(f"  {'-'*45}")
    for cat in cats:
        r = cnt_ru.get(cat, 0)
        e = cnt_en.get(cat, 0)
        delta = e - r
        sign = "+" if delta > 0 else ""
        print(f"  {cat:<18}  {r:>4}   {e:>4}   {sign}{delta:>+4}")

    print(f"\n  ИТОГО RU={len(vals_ru)}, EN={len(vals_en)}")

    # Матрица сопряжённости: строки=категории, столбцы=[RU, EN]
    obs = np.array([[cnt_ru.get(c, 0), cnt_en.get(c, 0)] for c in cats])

    # Убираем нулевые строки
    obs = obs[obs.sum(axis=1) > 0]

    if obs.shape[0] < 2:
        print("  ⚠ Слишком мало категорий для χ²")
        return

    # χ²
    chi2, p_chi2, dof, expected = chi2_contingency(obs, correction=False)
    chi2_yates, p_yates, _, _ = chi2_contingency(obs, correction=True)

    print(f"\n  χ² (без поправки): χ²={chi2:.4f}, df={dof}, p={p_chi2:.4f}  {'*' if p_chi2 < 0.05 else 'n.s.'}")
    print(f"  χ² (Yates):        χ²={chi2_yates:.4f}, df={dof}, p={p_yates:.4f}  {'*' if p_yates < 0.05 else 'n.s.'}")

    # Проверяем ожидаемые частоты для Fisher
    low_expected = (expected < 5).sum()
    total_cells  = expected.size
    print(f"\n  Ожидаемые частоты < 5: {low_expected}/{total_cells} ячеек")

    # Fisher для 2×2 если применимо
    if obs.shape[0] == 2:
        OR, p_fisher = fisher_exact(obs)
        print(f"  Fisher's exact: OR={OR:.4f}, p={p_fisher:.4f}  {'*' if p_fisher < 0.05 else 'n.s.'}")
    elif low_expected / total_cells > 0.2:
        print("  ⚠ >20% ячеек с ожидаемой частотой < 5 → результаты χ² ненадёжны")
        print("    Рекомендуется: объединить редкие категории или использовать permutation test")

    # Стандартизированные остатки для интерпретации
    print(f"\n  Стандартизированные остатки (|z|>2.0 = значимый вклад):")
    cats_valid = [c for c in cats if cnt_ru.get(c, 0) + cnt_en.get(c, 0) > 0]
    obs_v = np.array([[cnt_ru.get(c, 0), cnt_en.get(c, 0)] for c in cats_valid])
    _, _, _, exp_v = chi2_contingency(obs_v, correction=False)
    for i, cat in enumerate(cats_valid):
        for j, lang in enumerate(['RU', 'EN']):
            if exp_v[i, j] > 0:
                resid = (obs_v[i, j] - exp_v[i, j]) / np.sqrt(exp_v[i, j])
                if abs(resid) >= 1.5:
                    flag = " ← **" if abs(resid) >= 2.0 else ""
                    print(f"    {cat:<18} {lang}: z={resid:+.2f}{flag}")

    return {'chi2': chi2, 'p': p_chi2, 'dof': dof}


# Запускаем для каждой переменной
results = {}

results['тональность'] = contingency_analysis(
    'tone_ru', 'tone_en', paired,
    "ТОНАЛЬНОСТЬ: positive / neutral / negative — RU vs EN"
)

results['type'] = contingency_analysis(
    'type_ru', 'type_en', paired,
    "ТИП НОМИНАЦИИ: direct / metaphor / synesth / idiom — RU vs EN"
)

results['connotation'] = contingency_analysis(
    'conn_ru', 'conn_en', paired,
    "КОННОТАЦИЯ (семантическое поле): RU vs EN"
)


  ТОНАЛЬНОСТЬ: positive / neutral / negative — RU vs EN

  Категория          RU     EN    Δ(EN-RU)
  ---------------------------------------------
  negativ                4      4     +0
  negativ\neutral        1      1     +0
  neutral              153    153     +0
  positiv               12     12     +0

  ИТОГО RU=170, EN=170

  χ² (без поправки): χ²=0.0000, df=3, p=1.0000  n.s.
  χ² (Yates):        χ²=0.0000, df=3, p=1.0000  n.s.

  Ожидаемые частоты < 5: 4/8 ячеек
  ⚠ >20% ячеек с ожидаемой частотой < 5 → результаты χ² ненадёжны
    Рекомендуется: объединить редкие категории или использовать permutation test

  Стандартизированные остатки (|z|>2.0 = значимый вклад):

  ТИП НОМИНАЦИИ: direct / metaphor / synesth / idiom — RU vs EN

  Категория          RU     EN    Δ(EN-RU)
  ---------------------------------------------
  direct               162    162     +0
  metapher               1      1     +0
  metaphor               2      2     +0

  ИТОГО RU=165, EN=165

  χ² (без

## ЯЧЕЙКА 7 — Анализ cosine_sim по категориям ##

In [ ]:
print("\n" + "="*60)
print("COSINE SIMILARITY ПО КАТЕГОРИЯМ")
print("="*60)

import statistics

def group_stats(key, label):
    groups = defaultdict(list)
    for d in paired:
        val = d.get(key)
        if val:
            groups[val].append(d['cosine_sim'])
    print(f"\n  По {label}:")
    print(f"  {'Категория':<18} {'n':>4} {'mean':>7} {'median':>8} {'min':>7} {'max':>7}")
    print(f"  {'-'*55}")
    for k in sorted(groups.keys()):
        vals = groups[k]
        print(f"  {k:<18} {len(vals):>4} {np.mean(vals):>7.3f} {np.median(vals):>8.3f} {np.min(vals):>7.3f} {np.max(vals):>7.3f}")

group_stats('type_ru', 'type_RU (тип номинации)')
group_stats('tone_ru', 'тональность_RU')
group_stats('conn_ru', 'connotation_RU')

# Shift distribution
print(f"\n\nРАСПРЕДЕЛЕНИЕ translation_shift:")
shift_counts = Counter(d['shift_label'].split('+')[0] for d in paired)  # базовая категория
for k, v in sorted(shift_counts.items(), key=lambda x: -x[1]):
    pct = v / len(paired) * 100
    print(f"  {k:<22} {v:>3} ({pct:.1f}%)")

print(f"\nСтрок для ручной проверки (⚑): {sum(1 for d in paired if d['needs_review'])}")



COSINE SIMILARITY ПО КАТЕГОРИЯМ

  По type_RU (тип номинации):
  Категория             n    mean   median     min     max
  -------------------------------------------------------
  direct              162   0.734    0.762   0.121   0.973
  metapher              1   0.852    0.852   0.852   0.852
  metaphor              2   0.672    0.672   0.668   0.676

  По тональность_RU:
  Категория             n    mean   median     min     max
  -------------------------------------------------------
  negativ               4   0.567    0.621   0.354   0.672
  negativ\neutral       1   0.692    0.692   0.692   0.692
  neutral             153   0.739    0.766   0.121   0.973
  positiv              12   0.752    0.748   0.572   0.932

  По connotation_RU:
  Категория             n    mean   median     min     max
  -------------------------------------------------------
  духи                  5   0.533    0.569   0.303   0.680
  дым                   8   0.742    0.709   0.579   0.959
  еда     

## ЯЧЕЙКА 8 — Запись результатов в Excel ##

In [ ]:
# Стили
C_COMP_LIGHT = "F5EEF8"
C_ALT_COMP   = "EBE2F0"
HDR_COMP     = "4A235A"
C_FLAG_BG    = "FDEBD0"   # оранжевый фон для строк на проверку

def fill(h): return PatternFill("solid", start_color=h, fgColor=h)
def border():
    s = Side(style="thin", color="AAAAAA")
    return Border(left=s, right=s, top=s, bottom=s)

def write_cell(row, col, value, bg, bold=False, num_format=None):
    c = ws.cell(row=row, column=col)
    c.value = value
    c.font = Font(name="Arial", size=9, bold=bold)
    c.fill = fill(bg)
    c.alignment = Alignment(horizontal="center" if num_format else "left",
                            vertical="center", wrap_text=True)
    c.border = border()
    if num_format:
        c.number_format = num_format

def hdr(row, col, text):
    c = ws.cell(row=row, column=col)
    c.value = text
    c.font  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    c.fill  = fill(HDR_COMP)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border = border()

# Заголовки строки 2 (если пустые)
for col, label in [(C_SHIFT, "translation_shift"), (C_COS, "cosine_sim_LaBSE"),
                   (C_NOTES, "shift_notes"), (C_VERIFIED, "verified")]:
    if not ws.cell(2, col).value:
        hdr(2, col, label)

# Ширина колонок
from openpyxl.utils import get_column_letter
ws.column_dimensions[get_column_letter(C_SHIFT)].width    = 30
ws.column_dimensions[get_column_letter(C_COS)].width      = 16
ws.column_dimensions[get_column_letter(C_NOTES)].width    = 50
ws.column_dimensions[get_column_letter(C_VERIFIED)].width = 12

# Записываем данные
written = 0
for d in paired:
    r = d['row']
    is_even = ((r - 3) % 2 == 0)
    bg = C_ALT_COMP if is_even else C_COMP_LIGHT

    # Строки для ручной проверки — специальный фон
    if d['needs_review']:
        bg = C_FLAG_BG

    write_cell(r, C_SHIFT,    d['shift_label'],  bg)
    write_cell(r, C_COS,      round(d['cosine_sim'], 4), bg, num_format='0.0000')
    write_cell(r, C_NOTES,    d['shift_notes'],  bg)
    write_cell(r, C_VERIFIED, "⚑ REVIEW" if d['needs_review'] else "ok", bg,
               bold=d['needs_review'])
    written += 1

print(f"✓ Записано {written} строк в Excel")

# Сводный лист со статистикой
if 'Статистика' not in wb.sheetnames:
    ws_stat = wb.create_sheet('Статистика')
else:
    ws_stat = wb['Статистика']

# Очищаем и пишем сводку
ws_stat.delete_rows(1, ws_stat.max_row)

stat_rows = [
    ["LaBSE Cosine Similarity — сводка", "", ""],
    ["n пар", len(paired), ""],
    ["mean", round(np.mean(sims), 4), ""],
    ["median", round(np.median(sims), 4), ""],
    ["std", round(np.std(sims), 4), ""],
    ["min", round(np.min(sims), 4), ""],
    ["max", round(np.max(sims), 4), ""],
    ["≥0.90 (equivalent)", sum(1 for s in sims if s >= 0.90), f"{sum(1 for s in sims if s >= 0.90)/len(sims)*100:.1f}%"],
    ["0.75–0.89 (shift)", sum(1 for s in sims if 0.75 <= s < 0.90), f"{sum(1 for s in sims if 0.75 <= s < 0.90)/len(sims)*100:.1f}%"],
    ["<0.75 (significant_shift)", sum(1 for s in sims if s < 0.75), f"{sum(1 for s in sims if s < 0.75)/len(sims)*100:.1f}%"],
    ["", "", ""],
    ["Строки для ручной проверки", sum(1 for d in paired if d['needs_review']), ""],
    ["", "", ""],
    ["translation_shift — распределение", "", ""],
]
for k, v in sorted(shift_counts.items(), key=lambda x: -x[1]):
    stat_rows.append([k, v, f"{v/len(paired)*100:.1f}%"])

for i, row_data in enumerate(stat_rows, 1):
    for j, val in enumerate(row_data, 1):
        c = ws_stat.cell(row=i, column=j)
        c.value = val
        c.font = Font(name="Arial", size=10, bold=(j == 1 and i == 1))

print("✓ Лист 'Статистика' обновлён")


✓ Записано 172 строк в Excel
✓ Лист 'Статистика' обновлён


## ЯЧЕЙКА 9 — Сохранение и скачивание ##

In [ ]:
OUTPUT = "Разметка_сравнение_RU_EN_shifts.xlsx"
wb.save(OUTPUT)
print(f"✓ Файл сохранён: {OUTPUT}")

files.download(OUTPUT)
print("Скачивание началось.")
print()
print("=" * 60)
print("ИТОГОВОЕ РЕЗЮМЕ")
print("=" * 60)
print(f"Пар обработано:       {len(paired)}")
print(f"Среднее cosine sim:   {np.mean(sims):.4f}")
print(f"Equivalent (≥0.90):  {sum(1 for s in sims if s >= 0.90)}")
print(f"Shift (0.75–0.89):   {sum(1 for s in sims if 0.75 <= s < 0.90)}")
print(f"Sig. shift (<0.75):  {sum(1 for s in sims if s < 0.75)}")
print(f"На ручную проверку:  {sum(1 for d in paired if d['needs_review'])}")


✓ Файл сохранён: Разметка_сравнение_RU_EN_shifts.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Скачивание началось.

ИТОГОВОЕ РЕЗЮМЕ
Пар обработано:       172
Среднее cosine sim:   0.7372
Equivalent (≥0.90):  25
Shift (0.75–0.89):   66
Sig. shift (<0.75):  81
На ручную проверку:  82


# Этап 3


Статистический анализ семантических сдвигов при переводе ольфакторной лексики (RU→EN)
======================================================================================
Скрипт ищет статистически значимые зависимости, влияющие на степень сдвига,
и верифицирует результаты с помощью нескольких независимых методов.

Методы:
  1. Описательная статистика cosine_sim по всем категориальным переменным
  2. Kruskal-Wallis + попарные Mann-Whitney U (с поправкой Holm)
  3. Хи-квадрат + точный тест Фишера (для 2×2) — зависимость категорий
  4. Точечно-бисериальная корреляция (cosine_sim ~ бинарные флаги)
  5. Размер эффекта: η² (Kruskal-Wallis), Крамера V (хи-квадрат)
  6. Permutation test (10 000 перестановок) для верификации KW-результатов
  7. Логистическая регрессия: предикторы бинарного сдвига (significant_shift vs rest)
  8. Извлечение синтаксических признаков из gram_structure и их корреляция со сдвигом
  9. Визуализация: boxplot + violin + heatmap + forest plot (OR)
 10. Excel-отчёт со всеми таблицами результатов


## 3.0 Установка библиотек

In [ ]:
import re
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from itertools import combinations

from scipy import stats
from scipy.stats import (
    kruskal, mannwhitneyu, chi2_contingency, fisher_exact,
    pointbiserialr, spearmanr, pearsonr
)
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import Table2x2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils import resample

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# 3.0.1 Настройки

In [ ]:
INPUT_FILE  = "Разметка_сравнение_RU_EN_shifts_2604091729_.xlsx"
OUTPUT_XLS  = "shift_analysis_results.xlsx"
OUTPUT_FIG  = "shift_figures.png"
N_PERM      = 10_000
ALPHA       = 0.05
SEED        = 42
np.random.seed(SEED)

# 3.1 Загрузка и очистка

In [ ]:
print("=" * 68)
print("ЗАГРУЗКА ДАННЫХ")
print("=" * 68)

df = pd.read_excel(INPUT_FILE, sheet_name="Разметка", header=1)
df.columns = [str(c).strip() for c in df.columns]


# Нормализация строковых значений
str_cols = ["type_RU", "type_EN", "connotation_RU", "connotation_EN",
            "тональность_RU", "тональность_EN", "translation_shift"]
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower().replace("nan", np.nan)

# Исправляем опечатку "metapher"
df["type_RU"] = df["type_RU"].replace("metapher", "metaphor")
df["type_EN"] = df["type_EN"].replace("metapher", "metaphor")

# Нормализация тональности
df["тональность_RU"] = df["тональность_RU"].replace({
    "positiv": "positive", "negativ": "negative",
    "negativ\\neutral": "negative_neutral"
})
df["тональность_EN"] = df["тональность_EN"].replace({
    "positiv": "positive", "negativ": "negative",
    "negativ\\neutral": "negative_neutral"
})

# Числовой cosine_sim
df["cosine_sim"] = pd.to_numeric(df["cosine_sim_LaBSE"], errors="coerce")

# Порядковая переменная shift (0=equivalent, 1=shift, 2=significant_shift)
shift_order = {"equivalent": 0, "shift": 1, "significant_shift": 2}
df["shift_ord"] = df["translation_shift"].map(shift_order)

# Бинарная: significant vs остальные
df["is_sig"] = (df["translation_shift"] == "significant_shift").astype(int)
# Бинарная: any shift vs equivalent
df["is_any_shift"] = (df["translation_shift"] != "equivalent").astype(int)

df_clean = df.dropna(subset=["cosine_sim", "translation_shift"]).copy()
print(f"Строк с обеими переменными (cosine_sim + translation_shift): {len(df_clean)}")
print(f"Распределение сдвигов:\n{df_clean['translation_shift'].value_counts().to_string()}\n")


ЗАГРУЗКА ДАННЫХ
Строк с обеими переменными (cosine_sim + translation_shift): 172
Распределение сдвигов:
translation_shift
significant_shift    81
shift                66
equivalent           25



# 3.2 Извлечение синтаксических признаков из gram_structurу

In [ ]:
print("=" * 68)
print("ИЗВЛЕЧЕНИЕ СИНТАКСИЧЕСКИХ ПРИЗНАКОВ")
print("=" * 68)

def extract_syn_features(gram_str):
    """Извлекает набор бинарных синтаксических признаков из UD-строки."""
    if not gram_str or str(gram_str).strip() in ("", "nan"):
        return {}
    s = str(gram_str)
    return {
        "root_is_noun":   bool(re.search(r"ROOT\[NOUN\]", s)),
        "root_is_verb":   bool(re.search(r"ROOT\[VERB\]", s)),
        "root_is_adv":    bool(re.search(r"ROOT\[ADV\]", s)),
        "has_nsubj":      bool(re.search(r"\bnsubj\b", s)),
        "has_obj":        bool(re.search(r"\bobj\b", s)),
        "has_obl":        bool(re.search(r"\bobl\b", s)),
        "has_nmod":       bool(re.search(r"\bnmod\b", s)),
        "has_amod":       bool(re.search(r"\bamod\b", s)),
        "has_acl":        bool(re.search(r"\bacl\b", s)),
        "has_advmod":     bool(re.search(r"\badvmod\b", s)),
        "has_prep_of":    bool(re.search(r"prep\(of", s)),   # EN-специфично
        "has_conj":       bool(re.search(r"\bconj\b", s)),
        "depth_gte2":     s.count(">") >= 1,                 # есть вложенные зависимые
    }

for side, col in [("RU", "gram_structure"), ("EN", "gram.structure_EN")]:
    feats = df_clean[col].apply(extract_syn_features)
    feat_df = pd.DataFrame(list(feats), index=df_clean.index).fillna(False)
    feat_df.columns = [f"{side}_{c}" for c in feat_df.columns]
    df_clean = pd.concat([df_clean, feat_df], axis=1)

syn_cols_ru = [c for c in df_clean.columns if c.startswith("RU_")]
syn_cols_en = [c for c in df_clean.columns if c.startswith("EN_")]
print(f"Синтаксических признаков: RU={len(syn_cols_ru)}, EN={len(syn_cols_en)}")

# Признак структурного расхождения RU↔EN
df_clean["root_pos_mismatch"] = (
    df_clean["RU_root_is_noun"] != df_clean["EN_root_is_noun"]
).astype(int)
df_clean["depth_mismatch"] = (
    df_clean["RU_depth_gte2"] != df_clean["EN_depth_gte2"]
).astype(int)

# ─── Служебные функции ───────────────────────────────────────────────────────

def cohen_d(a, b):
    """Размер эффекта d для двух выборок."""
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    pooled = np.sqrt(((na - 1)*np.var(a, ddof=1) + (nb - 1)*np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else np.nan

def eta_squared_kruskal(H, k, n):
    """η² из статистики Краскела-Уоллиса."""
    return (H - k + 1) / (n - k)

def cramers_v(chi2, n, r, c):
    """V Крамера."""
    return np.sqrt(chi2 / (n * (min(r, c) - 1))) if (n * (min(r, c) - 1)) > 0 else 0.0

def permutation_kruskal(groups, n_perm=N_PERM):
    """Permutation test для Kruskal-Wallis."""
    all_vals = np.concatenate(groups)
    sizes    = [len(g) for g in groups]
    obs_H, _ = kruskal(*groups)
    count = 0
    for _ in range(n_perm):
        perm = np.random.permutation(all_vals)
        idx  = 0
        perm_groups = []
        for s in sizes:
            perm_groups.append(perm[idx:idx + s])
            idx += s
        H_perm, _ = kruskal(*perm_groups)
        if H_perm >= obs_H:
            count += 1
    return obs_H, count / n_perm

def kw_posthoc(groups, labels, alpha=ALPHA):
    """Попарные Mann-Whitney U с поправкой Holm."""
    pairs  = list(combinations(range(len(groups)), 2))
    pvals  = []
    ustats = []
    for i, j in pairs:
        u, p = mannwhitneyu(groups[i], groups[j], alternative="two-sided")
        pvals.append(p)
        ustats.append(u)
    reject, p_adj, _, _ = multipletests(pvals, method="holm")
    rows = []
    for idx, (i, j) in enumerate(pairs):
        d = cohen_d(groups[i], groups[j])
        rows.append({
            "group_A": labels[i], "group_B": labels[j],
            "n_A": len(groups[i]), "n_B": len(groups[j]),
            "U": ustats[idx], "p_raw": pvals[idx],
            "p_holm": p_adj[idx], "sig": "*" if reject[idx] else "n.s.",
            "cohen_d": round(d, 3) if not np.isnan(d) else np.nan,
        })
    return pd.DataFrame(rows)

ИЗВЛЕЧЕНИЕ СИНТАКСИЧЕСКИХ ПРИЗНАКОВ
Синтаксических признаков: RU=13, EN=13


# 3.3 Kruskal-Wallis: cosine_sim ~ категориальные предикторы

In [ ]:
results_kw = []

def run_kruskal(col, label, df=df_clean):
    col_vals = df[col].dropna()
    cats = [c for c in col_vals.unique() if str(c) not in ("nan", "none", "")]
    groups = {c: df.loc[df[col] == c, "cosine_sim"].dropna().values for c in cats}
    groups = {k: v for k, v in groups.items() if len(v) >= 3}
    if len(groups) < 2:
        return None, None
    g_list = list(groups.values())
    g_keys = list(groups.keys())
    H, p   = kruskal(*g_list)
    n_tot  = sum(len(g) for g in g_list)
    eta2   = eta_squared_kruskal(H, len(g_list), n_tot)
    _, p_perm = permutation_kruskal(g_list)
    row = {
        "Переменная": label, "H": round(H, 3), "df": len(g_list) - 1,
        "p_kruskal": round(p, 4), "p_perm": round(p_perm, 4),
        "eta²": round(eta2, 3), "sig_kw": "*" if p < ALPHA else "n.s.",
        "sig_perm": "*" if p_perm < ALPHA else "n.s.", "n": n_tot,
    }
    results_kw.append(row)
    posthoc = kw_posthoc(g_list, [str(k) for k in g_keys])
    return row, posthoc

categorical_vars = [
    ("type_RU",         "Тип номинации RU"),
    ("connotation_RU",  "Семантическое поле RU"),
    ("тональность_RU",  "Тональность RU"),
    ("type_EN",         "Тип номинации EN"),
    ("connotation_EN",  "Семантическое поле EN"),
    ("тональность_EN",  "Тональность EN"),
]

posthoc_tables = {}
print("\nKRUSKAL-WALLIS: cosine_sim ~ категориальные предикторы")
print(f"{'Переменная':<28} {'H':>7} {'df':>3} {'p_kw':>8} {'p_perm':>8} {'η²':>7} {'sig':>5}")
print("-" * 70)
for col, label in categorical_vars:
    row, ph = run_kruskal(col, label)
    if row:
        print(f"{label:<28} {row['H']:>7.3f} {row['df']:>3} {row['p_kruskal']:>8.4f} "
              f"{row['p_perm']:>8.4f} {row['eta²']:>7.3f} {row['sig_kw']:>5}")
        posthoc_tables[label] = ph

df_kw = pd.DataFrame(results_kw)


KRUSKAL-WALLIS: cosine_sim ~ категориальные предикторы
Переменная                         H  df     p_kw   p_perm      η²   sig
----------------------------------------------------------------------
Тип номинации RU               0.143   1   0.7054   0.7189  -0.005  n.s.
Семантическое поле RU         21.631  10   0.0171   0.0101   0.120     *
Тональность RU                 5.182   2   0.0749   0.0727   0.019  n.s.
Тип номинации EN               0.143   1   0.7054   0.7209  -0.005  n.s.
Семантическое поле EN         21.631  10   0.0171   0.0092   0.120     *
Тональность EN                 5.182   2   0.0749   0.0743   0.019  n.s.


# 3.3.1 Post-hoc таблицы

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── 1. Вывод таблицы ────────────────────────────────────────────────────────
key = "Семантическое поле RU"
ph = posthoc_tables[key]

print(f"\nPOST-HOC: {key}")
print(f"{'Группа A':<22} {'Группа B':<22} {'n_A':>5} {'n_B':>5} "
      f"{'U':>8} {'p_raw':>8} {'p_holm':>8} {'sig':>5} {'d':>7}")
print("-" * 95)

for _, row in ph.sort_values("p_holm").iterrows():
    print(f"{row['group_A']:<22} {row['group_B']:<22} "
          f"{int(row['n_A']):>5} {int(row['n_B']):>5} "
          f"{row['U']:>8.1f} {row['p_raw']:>8.4f} "
          f"{row['p_holm']:>8.4f} {row['sig']:>5} "
          f"{row['cohen_d']:>7.3f}")

# ── 2. Сводка только значимых пар ───────────────────────────────────────────
sig_pairs = ph[ph["sig"] == "*"].sort_values("p_holm")
print(f"\nЗначимых пар: {len(sig_pairs)} из {len(ph)}")
if not sig_pairs.empty:
    print(sig_pairs[["group_A", "group_B", "p_holm", "cohen_d"]].to_string(index=False))

# ── 3. Heatmap p-значений ────────────────────────────────────────────────────
groups_list = sorted(set(ph["group_A"]) | set(ph["group_B"]))
n = len(groups_list)
idx = {g: i for i, g in enumerate(groups_list)}

# Матрица p-значений (Holm)
pmat = np.ones((n, n))
dmat = np.zeros((n, n))

for _, row in ph.iterrows():
    i, j = idx[row["group_A"]], idx[row["group_B"]]
    pmat[i, j] = pmat[j, i] = row["p_holm"]
    dmat[i, j] = dmat[j, i] = abs(row["cohen_d"]) if not np.isnan(row["cohen_d"]) else 0

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"Post-hoc анализ: {key}", fontsize=14, fontweight="bold")

# — Heatmap p-value —
mask_diag = np.eye(n, dtype=bool)
annot_p = np.where(mask_diag, "",
           np.where(pmat < 0.05,
                    np.vectorize(lambda x: f"{x:.3f}*")(pmat),
                    np.vectorize(lambda x: f"{x:.3f}")(pmat)))

sns.heatmap(
    pmat, ax=axes[0],
    xticklabels=groups_list, yticklabels=groups_list,
    annot=annot_p, fmt="", cmap="RdYlGn_r",
    vmin=0, vmax=0.1, mask=mask_diag,
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "p (Holm)"}
)
axes[0].set_title("p-значения (Holm)\n* = p < 0.05", fontsize=11)
axes[0].tick_params(axis="x", rotation=45)
axes[0].tick_params(axis="y", rotation=0)

# — Heatmap cohen_d —
annot_d = np.where(mask_diag, "",
           np.where(pmat < 0.05,
                    np.vectorize(lambda x: f"{x:.2f}*")(dmat),
                    np.vectorize(lambda x: f"{x:.2f}")(dmat)))

sns.heatmap(
    dmat, ax=axes[1],
    xticklabels=groups_list, yticklabels=groups_list,
    annot=annot_d, fmt="", cmap="Blues",
    vmin=0, mask=mask_diag,
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "|cohen d|"}
)
axes[1].set_title("Размер эффекта |cohen d|\n* = значимая пара", fontsize=11)
axes[1].tick_params(axis="x", rotation=45)
axes[1].tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("posthoc_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: posthoc_heatmap.png")

# ── 4. Боксплот по группам ───────────────────────────────────────────────────
col = "connotation_RU"   # исходная колонка

group_order = (df_clean.groupby(col)["cosine_sim"]
               .median()
               .sort_values(ascending=False)
               .index.tolist())

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df_clean, x=col, y="cosine_sim",
            order=group_order, palette="Set2", ax=ax)
sns.stripplot(data=df_clean, x=col, y="cosine_sim",
              order=group_order, color="black", alpha=0.3, size=3, ax=ax)

ax.set_title(f"cosine_sim по группам: {key}", fontsize=13)
ax.set_xlabel("")
ax.set_ylabel("cosine_sim")
ax.tick_params(axis="x", rotation=40)

# Подписи значимых пар прямо на графике
if not sig_pairs.empty:
    ax.set_title(
        f"cosine_sim по группам: {key}\n"
        f"Значимых пар (post-hoc Holm): {len(sig_pairs)}",
        fontsize=12
    )

plt.tight_layout()
plt.savefig("posthoc_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: posthoc_boxplot.png")


POST-HOC: Семантическое поле RU
Группа A               Группа B                 n_A   n_B        U    p_raw   p_holm   sig       d
-----------------------------------------------------------------------------------------------
еда                    духи                      25     5    122.0   0.0001   0.0054     *   2.420
природа                духи                      33     5    151.0   0.0013   0.0697  n.s.   1.816
цветы                  духи                      10     5     47.0   0.0047   0.2471  n.s.   2.075
еда                    оружие                    25     5    107.0   0.0108   0.5612  n.s.   1.166
природа                оружие                    33     5    129.0   0.0445   1.0000  n.s.   0.631
природа                товары                    33     8    159.0   0.3906   1.0000  n.s.   0.923
природа                одежда                    33     3     37.0   0.5120   1.0000  n.s.  -0.431
природа                еда                       33    25    345.5   0.2964   1


# 3.4 Хи-квадрат: shift_category ~ категориальные предикторы

In [ ]:
results_chi = []

def run_chi2(col, label, df=df_clean):
    sub = df[[col, "translation_shift"]].dropna()
    sub = sub[sub[col].astype(str).str.lower() not in ["nan", ""]]
    sub = sub[sub[col].notna()]
    ctab = pd.crosstab(sub[col], sub["translation_shift"])
    if ctab.shape[0] < 2 or ctab.shape[1] < 2:
        return None
    chi2, p, dof, exp = chi2_contingency(ctab, correction=False)
    n = ctab.values.sum()
    V = cramers_v(chi2, n, ctab.shape[0], ctab.shape[1])
    low_exp = (exp < 5).sum() / exp.size
    row = {
        "Переменная": label, "χ²": round(chi2, 3), "df": dof,
        "p": round(p, 4), "Крамера V": round(V, 3),
        "sig": "*" if p < ALPHA else "n.s.",
        ">20% ожид.<5": "да" if low_exp > 0.2 else "нет",
        "n": n,
    }
    results_chi.append(row)
    return ctab

chi2_crosstabs = {}
print("\n\nХИ-КВАДРАТ: translation_shift ~ категориальные предикторы")
print(f"{'Переменная':<28} {'χ²':>8} {'df':>3} {'p':>8} {'V':>7} {'sig':>5}")
print("-" * 58)
for col, label in categorical_vars:
    ct = run_chi2(col, label)
    if ct is not None:
        r = results_chi[-1]
        print(f"{label:<28} {r['χ²']:>8.3f} {r['df']:>3} {r['p']:>8.4f} {r['Крамера V']:>7.3f} {r['sig']:>5}")
        chi2_crosstabs[label] = ct

df_chi = pd.DataFrame(results_chi)



ХИ-КВАДРАТ: translation_shift ~ категориальные предикторы
Переменная                         χ²  df        p       V   sig
----------------------------------------------------------


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:


# ─── 5. Синтаксические признаки: точечно-бисериальная корреляция ─────────────

print("\n\nТОЧЕЧНО-БИСЕРИАЛЬНАЯ КОРРЕЛЯЦИЯ: синтаксич. признаки ~ cosine_sim")
print(f"{'Признак':<30} {'r_pb':>8} {'p':>8} {'sig':>5} {'% True':>8}")
print("-" * 60)

pb_results = []
all_syn_cols = syn_cols_ru + syn_cols_en + ["root_pos_mismatch", "depth_mismatch"]
for col in all_syn_cols:
    vals = df_clean[col].astype(float)
    sims = df_clean["cosine_sim"]
    mask = vals.notna() & sims.notna()
    if mask.sum() < 10 or vals[mask].nunique() < 2:
        continue
    r, p = pointbiserialr(vals[mask], sims[mask])
    pct  = vals[mask].mean() * 100
    pb_results.append({"Признак": col, "r_pb": round(r, 3), "p": round(p, 4),
                        "sig": "*" if p < ALPHA else "n.s.", "%_True": round(pct, 1)})
    if p < ALPHA:
        print(f"{col:<30} {r:>8.3f} {p:>8.4f} {'*':>5} {pct:>7.1f}%")

df_pb = pd.DataFrame(pb_results).sort_values("p")

# ─── 6. Корреляция shift_ord ~ синтаксические признаки (Spearman) ────────────

print("\n\nSPEARMAN: shift_ord (порядковый сдвиг) ~ синтаксические признаки")
print(f"{'Признак':<30} {'ρ':>8} {'p':>8} {'sig':>5}")
print("-" * 52)

sp_results = []
for col in all_syn_cols:
    vals = df_clean[col].astype(float)
    ord_ = df_clean["shift_ord"]
    mask = vals.notna() & ord_.notna()
    if mask.sum() < 10 or vals[mask].nunique() < 2:
        continue
    rho, p = spearmanr(vals[mask], ord_[mask])
    sp_results.append({"Признак": col, "ρ": round(rho, 3), "p": round(p, 4),
                        "sig": "*" if p < ALPHA else "n.s."})
    if p < ALPHA:
        print(f"{col:<30} {rho:>8.3f} {p:>8.4f} {'*':>5}")

df_sp = pd.DataFrame(sp_results).sort_values("p")

# ─── 7. Логистическая регрессия ──────────────────────────────────────────────

print("\n\nЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ: предикторы significant_shift")

# Признаки
cat_feat_cols = ["type_RU", "connotation_RU", "тональность_RU"]
syn_feat_cols = all_syn_cols

lr_df = df_clean[cat_feat_cols + syn_feat_cols + ["is_sig"]].copy()

# One-hot для категорий
lr_df = pd.get_dummies(lr_df, columns=cat_feat_cols, drop_first=True)
lr_df = lr_df.dropna()
lr_df = lr_df.astype(float)

X = lr_df.drop("is_sig", axis=1)
y = lr_df["is_sig"]

if len(y) >= 30 and y.nunique() == 2:
    clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", random_state=SEED)
    clf.fit(X, y)
    y_pred = clf.predict(X)
    y_prob = clf.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_prob)
    print(f"\n  ROC-AUC (in-sample): {auc:.3f}")
    print(classification_report(y, y_pred, target_names=["not_sig", "significant_shift"]))

    # Bootstrap CI для коэффициентов
    coef_boots = []
    for _ in range(1000):
        idx = resample(range(len(X)), random_state=None)
        Xb, yb = X.iloc[idx], y.iloc[idx]
        if yb.nunique() < 2:
            continue
        m = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", random_state=SEED)
        m.fit(Xb, yb)
        coef_boots.append(m.coef_[0])
    coef_boots = np.array(coef_boots)
    ci_lo = np.percentile(coef_boots, 2.5, axis=0)
    ci_hi = np.percentile(coef_boots, 97.5, axis=0)

    lr_coef = pd.DataFrame({
        "Feature": X.columns,
        "Coefficient": clf.coef_[0],
        "OR": np.exp(clf.coef_[0]),
        "CI_lo (OR)": np.exp(ci_lo),
        "CI_hi (OR)": np.exp(ci_hi),
    }).sort_values("Coefficient", ascending=False)
    print("\n  Топ-10 предикторов (по |коэф.|):")
    top10 = lr_coef.reindex(lr_coef["Coefficient"].abs().sort_values(ascending=False).index).head(10)
    print(top10[["Feature", "Coefficient", "OR", "CI_lo (OR)", "CI_hi (OR)"]].to_string(index=False))
else:
    lr_coef = pd.DataFrame()
    print("  Недостаточно данных для логистической регрессии.")

# ─── 8. Визуализация ─────────────────────────────────────────────────────────

print("\n\nСОЗДАНИЕ ГРАФИКОВ...")

palette = {"equivalent": "#4CAF50", "shift": "#FF9800", "significant_shift": "#F44336"}
sns.set_style("whitegrid")
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle("Семантический сдвиг при переводе ольфакторной лексики RU→EN\nСтатистический анализ",
             fontsize=14, fontweight="bold", y=1.01)

# 8a. Violin: cosine_sim по shift
ax = axes[0, 0]
order = ["equivalent", "shift", "significant_shift"]
sns.violinplot(data=df_clean, x="translation_shift", y="cosine_sim",
               order=order, palette=palette, ax=ax, inner="box")
ax.set_title("Cosine similarity по степени сдвига")
ax.set_xlabel(""); ax.set_ylabel("LaBSE cosine sim")
for i, cat in enumerate(order):
    vals = df_clean.loc[df_clean["translation_shift"] == cat, "cosine_sim"].dropna()
    ax.text(i, vals.min() - 0.04, f"n={len(vals)}", ha="center", fontsize=8)

# 8b. Boxplot: cosine_sim по тональности
ax = axes[0, 1]
tone_order = sorted(df_clean["тональность_RU"].dropna().unique())
sns.boxplot(data=df_clean, x="тональность_RU", y="cosine_sim",
            order=tone_order, ax=ax)
ax.set_title("Cosine sim по тональности RU")
ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(axis="x", rotation=20)

# 8c. Boxplot: cosine_sim по семантическому полю (top-8 категорий)
ax = axes[0, 2]
top_conn = df_clean["connotation_RU"].value_counts().head(8).index.tolist()
sub_conn = df_clean[df_clean["connotation_RU"].isin(top_conn)]
sns.boxplot(data=sub_conn, x="connotation_RU", y="cosine_sim", order=top_conn, ax=ax)
ax.set_title("Cosine sim по семантическому полю RU (top-8)")
ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)

# 8d. Heatmap: семантическое поле × translation_shift
ax = axes[1, 0]
ct = pd.crosstab(df_clean["connotation_RU"], df_clean["translation_shift"])
ct_norm = ct.div(ct.sum(axis=1), axis=0)
sns.heatmap(ct_norm, annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Доля каждого типа сдвига\nпо семантическому полю")
ax.set_xlabel(""); ax.set_ylabel("Сем. поле")
ax.tick_params(axis="x", rotation=30)

# 8e. Point-biserial топ значимых признаков
ax = axes[1, 1]
sig_pb = df_pb[df_pb["sig"] == "*"].copy()
if not sig_pb.empty:
    sig_pb = sig_pb.sort_values("r_pb")
    colors = ["#F44336" if r < 0 else "#4CAF50" for r in sig_pb["r_pb"]]
    ax.barh(sig_pb["Признак"], sig_pb["r_pb"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title("Знач. корр. (r_pb) синтакс. признаки ~ cosine_sim")
    ax.set_xlabel("Point-biserial r")
else:
    ax.text(0.5, 0.5, "Нет значимых признаков", ha="center", va="center",
            transform=ax.transAxes)
    ax.set_title("Синтаксические признаки ~ cosine_sim")

# 8f. Forest plot OR из логистической регрессии
ax = axes[1, 2]
if not lr_coef.empty:
    top_lr = lr_coef.reindex(lr_coef["Coefficient"].abs().sort_values(ascending=False).index).head(12)
    top_lr = top_lr.sort_values("Coefficient")
    ax.barh(range(len(top_lr)), top_lr["Coefficient"],
            xerr=[top_lr["Coefficient"] - np.log(top_lr["CI_lo (OR)"]),
                  np.log(top_lr["CI_hi (OR)"]) - top_lr["Coefficient"]],
            color=["#F44336" if c > 0 else "#4CAF50" for c in top_lr["Coefficient"]],
            capsize=3, alpha=0.8)
    ax.set_yticks(range(len(top_lr)))
    ax.set_yticklabels(top_lr["Feature"], fontsize=7)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title("LR: коэффициенты (± 95% CI bootstrap)\nпредиктор: significant_shift")
    ax.set_xlabel("log(OR)")
else:
    ax.set_visible(False)

# 8g. Распределение cosine_sim с разбивкой
ax = axes[2, 0]
for cat in order:
    vals = df_clean.loc[df_clean["translation_shift"] == cat, "cosine_sim"].dropna()
    ax.hist(vals, bins=20, alpha=0.55, label=cat, color=palette[cat])
ax.axvline(0.75, color="blue", linestyle="--", linewidth=1, label="порог 0.75")
ax.axvline(0.90, color="green", linestyle="--", linewidth=1, label="порог 0.90")
ax.set_title("Гистограмма cosine_sim по типу сдвига")
ax.set_xlabel("LaBSE cosine sim"); ax.set_ylabel("Кол-во")
ax.legend(fontsize=8)

# 8h. Матрица корреляций (Spearman): топ-8 синт. признаков × shift_ord + cosine
ax = axes[2, 1]
top_syn = df_sp.head(8)["Признак"].tolist() if not df_sp.empty else all_syn_cols[:8]
corr_cols = top_syn + ["shift_ord", "cosine_sim"]
corr_cols = [c for c in corr_cols if c in df_clean.columns]
corr_mat = df_clean[corr_cols].dropna().astype(float).corr(method="spearman")
mask = np.zeros_like(corr_mat, dtype=bool)
np.fill_diagonal(mask, True)
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, cbar_kws={"shrink": 0.7},
            annot_kws={"size": 7})
ax.set_title("Spearman корреляции:\nсинтакс. признаки × shift_ord")
ax.tick_params(axis="both", labelsize=7)

# 8i. Stacked bar: тип номинации × shift
ax = axes[2, 2]
ct2 = pd.crosstab(df_clean["type_RU"], df_clean["translation_shift"])[
    [c for c in order if c in df_clean["translation_shift"].unique()]
]
ct2_norm = ct2.div(ct2.sum(axis=1), axis=0)
ct2_norm.plot(kind="bar", stacked=True, ax=ax,
              color=[palette[c] for c in ct2_norm.columns])
ax.set_title("Тип номинации RU × степень сдвига")
ax.set_xlabel(""); ax.legend(loc="upper right", fontsize=7)
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_FIG, dpi=150, bbox_inches="tight")
print(f"  → Сохранено: {OUTPUT_FIG}")

# ─── 9. Excel-отчёт ──────────────────────────────────────────────────────────

print("\n\nСОЗДАНИЕ EXCEL-ОТЧЁТА...")

wb = openpyxl.Workbook()
wb.remove(wb.active)

HDR_BG  = "2C3E50"
HDR_FG  = "FFFFFF"
SIG_BG  = "FDEBD0"
GOOD_BG = "D5F5E3"
WARN_BG = "FDFEFE"

def _border():
    s = Side(style="thin", color="CCCCCC")
    return Border(left=s, right=s, top=s, bottom=s)

def hdr_cell(ws, row, col, text):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(name="Arial", bold=True, color=HDR_FG, size=10)
    c.fill = PatternFill("solid", fgColor=HDR_BG)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border = _border()

def data_cell(ws, row, col, value, bg=None, bold=False):
    c = ws.cell(row=row, column=col, value=value)
    c.font = Font(name="Arial", size=9, bold=bold)
    if bg:
        c.fill = PatternFill("solid", fgColor=bg)
    c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    c.border = _border()

def write_df_to_sheet(wb, df_out, sheet_name, sig_col=None, sig_val="*"):
    ws = wb.create_sheet(sheet_name)
    for ci, col in enumerate(df_out.columns, 1):
        hdr_cell(ws, 1, ci, col)
        ws.column_dimensions[get_column_letter(ci)].width = max(14, len(str(col)) + 2)
    for ri, row in df_out.iterrows():
        is_sig = sig_col and str(row.get(sig_col, "")) == sig_val
        bg = SIG_BG if is_sig else None
        for ci, val in enumerate(row.values, 1):
            v = round(val, 4) if isinstance(val, float) else val
            data_cell(ws, ri + 2, ci, v, bg=bg, bold=is_sig)
    ws.freeze_panes = "A2"
    return ws

# Лист 0: Сводка
ws0 = wb.create_sheet("0_Сводка")
summary = [
    ["АНАЛИЗ СЕМАНТИЧЕСКИХ СДВИГОВ ПРИ ПЕРЕВОДЕ ОЛЬФАКТОРНОЙ ЛЕКСИКИ RU→EN", ""],
    ["", ""],
    ["ОБЩАЯ СТАТИСТИКА", ""],
    ["Всего пар (с cosine_sim)", len(df_clean)],
    ["equivalent", (df_clean["translation_shift"] == "equivalent").sum()],
    ["shift", (df_clean["translation_shift"] == "shift").sum()],
    ["significant_shift", (df_clean["translation_shift"] == "significant_shift").sum()],
    ["Среднее cosine_sim", round(df_clean["cosine_sim"].mean(), 4)],
    ["Медиана cosine_sim", round(df_clean["cosine_sim"].median(), 4)],
    ["Std cosine_sim", round(df_clean["cosine_sim"].std(), 4)],
    ["", ""],
    ["КЛЮЧЕВЫЕ НАХОДКИ", ""],
    ["Методы", "Kruskal-Wallis + Permutation test (10k), χ², LR с bootstrap CI"],
    ["Уровень α", ALPHA],
]
for i, row_data in enumerate(summary, 1):
    for j, val in enumerate(row_data, 1):
        c = ws0.cell(row=i, column=j, value=val)
        c.font = Font(name="Arial", bold=(j == 1 and i in [1, 3, 12]), size=10)
ws0.column_dimensions["A"].width = 40
ws0.column_dimensions["B"].width = 20

# Листы с результатами
write_df_to_sheet(wb, df_kw, "1_KruskalWallis", sig_col="sig_kw")
write_df_to_sheet(wb, df_chi, "2_Хи-квадрат", sig_col="sig")
write_df_to_sheet(wb, df_pb, "3_PointBiserial", sig_col="sig")
write_df_to_sheet(wb, df_sp, "4_Spearman_ShiftOrd", sig_col="sig")

if not lr_coef.empty:
    lr_out = lr_coef.round(4)
    write_df_to_sheet(wb, lr_out, "5_LogisticRegression")

# Post-hoc таблицы
for label, ph_df in posthoc_tables.items():
    if ph_df is not None and not ph_df.empty:
        short = label[:24]
        write_df_to_sheet(wb, ph_df.round(4), f"PH_{short}", sig_col="sig")

# Кросстаблицы (нормализованные)
for label, ct in chi2_crosstabs.items():
    ct_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
    ws_ct = wb.create_sheet(f"CT_{label[:20]}")
    hdr_cell(ws_ct, 1, 1, label)
    for ci, col in enumerate(ct_pct.columns, 2):
        hdr_cell(ws_ct, 1, ci, col)
    for ri, (idx, row) in enumerate(ct_pct.iterrows(), 2):
        data_cell(ws_ct, ri, 1, idx)
        for ci, val in enumerate(row.values, 2):
            bg = SIG_BG if val > 60 else (GOOD_BG if val > 40 else None)
            data_cell(ws_ct, ri, ci, val, bg=bg)

wb.save(OUTPUT_XLS)
print(f"  → Сохранено: {OUTPUT_XLS}")

# ─── 10. Финальный вывод ─────────────────────────────────────────────────────

print("\n" + "=" * 68)
print("ИТОГОВОЕ РЕЗЮМЕ ЗНАЧИМЫХ ЗАВИСИМОСТЕЙ")
print("=" * 68)

sig_kw = df_kw[df_kw["sig_kw"] == "*"] if not df_kw.empty else pd.DataFrame()
sig_chi = df_chi[df_chi["sig"] == "*"] if not df_chi.empty else pd.DataFrame()
sig_pb_f = df_pb[df_pb["sig"] == "*"] if not df_pb.empty else pd.DataFrame()
sig_sp_f = df_sp[df_sp["sig"] == "*"] if not df_sp.empty else pd.DataFrame()

print(f"\n[Kruskal-Wallis] Значимых предикторов cosine_sim: {len(sig_kw)}")
for _, r in sig_kw.iterrows():
    print(f"  ✓ {r['Переменная']}: H={r['H']}, p={r['p_kruskal']}, η²={r['eta²']}")

print(f"\n[χ²] Значимых зависимостей от типа сдвига: {len(sig_chi)}")
for _, r in sig_chi.iterrows():
    print(f"  ✓ {r['Переменная']}: χ²={r['χ²']}, p={r['p']}, V={r['Крамера V']}")

print(f"\n[Point-biserial] Синтакс. признаков, коррел. с cosine_sim: {len(sig_pb_f)}")
for _, r in sig_pb_f.iterrows():
    print(f"  ✓ {r['Признак']}: r={r['r_pb']}, p={r['p']}")

print(f"\n[Spearman] Синтакс. признаков, коррел. с порядковым сдвигом: {len(sig_sp_f)}")
for _, r in sig_sp_f.iterrows():
    print(f"  ✓ {r['Признак']}: ρ={r['ρ']}, p={r['p']}")

print(f"\nФайлы сохранены:\n  {OUTPUT_XLS}\n  {OUTPUT_FIG}")
print("=" * 68)
